In [ ]:
"""

Apply the VALSO function to output from IPSL-CM7

"""

In [1]:
import basal_melt_NEMO.metrics_functions as metf
import xarray as xr
import os
import glob
import numpy as np

In [2]:
inputpath = '/thredds/tgcc/store/p86caub/IPSLCM7/DEVT/pdControl/'
mask_path = '/data/cburgard/TOOLS/'

In [3]:
### MASK FILE

In [4]:
ocean_masks = xr.open_dataset(mask_path + 'basin_masks_orca1_nemo4p2.nc')

In [5]:
### VAR LIST

In [6]:
#var_ocean_list_in = ['global_SST', 'wed_gyre','ross_gyre','ACC',
#            #'FRIS_melt','Ross_melt','LarsenC_melt','total_melt',
#            'Sbot_WWED','Sbot_EWED','Sbot_WROSS','Sbot_EROSS','Sbot_AMU',
#            'Tbot_WWED','Tbot_EWED','Tbot_WROSS','Tbot_EROSS','Tbot_AMU',
#            'OHC_tot','OHC_700','OHC_2000',
#            'AMOC_26N','AMOC_30S']

#var_seaice_list_in = ['mar_sie_arc','sep_sie_arc','feb_sie_ant','sep_sie_ant',
#                      'mar_sia_arc','sep_sia_arc','feb_sia_ant','sep_sia_ant',
#                     'mar_siv_arc','sep_siv_arc','feb_siv_ant','sep_siv_ant']

In [17]:
var_ocean_list_in = ['global_SST', 
            #'FRIS_melt','Ross_melt','LarsenC_melt','total_melt',
            'Sbot_WWED','Sbot_EWED','Sbot_WROSS','Sbot_EROSS','Sbot_AMU',
           'Tbot_WWED','Tbot_EWED','Tbot_WROSS','Tbot_EROSS','Tbot_AMU']
           # 'OHC_tot','OHC_700','OHC_2000']

run_list = ['CM70-O4-pd-nnetau1-01','CM70-ico-ICO60-O4-pd-01']

In [15]:
ds_list = []
area_list = []
for rrun in run_list:
    inputpath2 = inputpath + rrun + '/OCE/Analyse/TS_MO/'

    files = glob.glob(inputpath2+'*thetao*')
    files_sorted = sorted(files, key=os.path.getmtime, reverse=True)
    most_recent_file = files[0]

    ds_run_theta = xr.open_mfdataset(most_recent_file)['thetao']

    files = glob.glob(inputpath2+'*so.nc')
    files_sorted = sorted(files, key=os.path.getmtime, reverse=True)
    most_recent_file = files[0]

    ds_run_sal = xr.open_mfdataset(most_recent_file)['so']
    
    ds_list.append(xr.merge([ds_run_theta, ds_run_sal]).assign_coords({'run': rrun}).rename({'time_counter': 'time'}))

    inputpath3 = inputpath + rrun + '/OCE/Analyse/SE/'

    ds_area = xr.open_mfdataset(inputpath3 + rrun + '_SE_1890_1899_1M_grid_T.nc', decode_times=False)['area']
    area_list.append(ds_area)
    
#ds_T_all_runs = xr.concat(ds_list, dim='run')
    

/home/cburgard/.conda/envs/py38/lib/python3.8/site-packages/xarray/core/indexing.py:1374: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


In [ ]:
var_ds_all = metf.compute_VALSO_ocean_metrics(var_ocean_list_in, 
                                       run_list, 
                                      ds_list, 
                                      ds_list, 
                                      ds_list, 
                                      area_list, 
                                         ocean_masks, 
                                         smallest_domain=True)

f = metf.VALSO_plot_ocean_comparison(var_ocean_list_in, 
                        var_ds_all,
                      run_list, 
                      ['deepskyblue','royalblue'])
f.show()

USING CELL AREA FROM RUN NUMBER 1 IN THE LIST
CM70-O4-pd-nnetau1-01
Computing global SST
Computing gyres
Computing bottom properties
Computing ocean heat content
CM70-ico-ICO60-O4-pd-01
Computing global SST
Computing gyres
Computing bottom properties


In [ ]:
var_ds_seaice = mf.compute_VALSO_seaice_metrics(var_seaice_list_in, 
                                        ['closed_isf','open_isf'], 
                                        [file_T_closed_all, file_T_open_all], 
                                        [file_ice_closed_all, file_ice_open_all], 
                                        [cellarea_closed, cellarea_open], 
                                        ['deepskyblue','royalblue'],
                                        smallest_domain=True)

f_seaice = mf.VALSO_plot_seaice_comparison(var_seaice_list_in, 
                                        var_ds_seaice, 
                                        ['closed_isf','open_isf'],  
                                        ['deepskyblue','royalblue'])